In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
import os, json
import numpy as np
import torch

In [3]:
def load_jsonl(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            rows.append(json.loads(line))
    return rows

absa_train_path = "/content/drive/MyDrive/FYP/absa/absa_train_big.jsonl"
absa_dev_path = "/content/drive/MyDrive/FYP/absa/absa_dev.jsonl"
absa_test_path = "/content/drive/MyDrive/FYP/absa/absa_test.jsonl"

train_rows = load_jsonl(absa_train_path)
dev_rows = load_jsonl(absa_dev_path)
test_rows = load_jsonl(absa_test_path)

In [4]:
LABELS = ["negative", "neutral", "positive", "conflict"]
label2id = {lab:i for i, lab in enumerate(LABELS)}
id2label = {i:lab for lab, i in label2id.items()}

In [5]:
def norm_pol(p):
    return p.strip().lower()

In [6]:
def expand_absa(rows):
    out_text = []
    out_cat  = []
    out_lab  = []
    for r in rows:
        text = r["text"]
        for lab in r["labels"]:
            cat = lab["category"]
            pol = norm_pol(lab["polarity"])
            if pol not in label2id:
                continue
            out_text.append(text)
            out_cat.append(cat)
            out_lab.append(label2id[pol])
    return {"text": out_text, "category": out_cat, "label": out_lab}

In [7]:
from datasets import Dataset
train_ds = Dataset.from_dict(expand_absa(train_rows))
dev_ds = Dataset.from_dict(expand_absa(dev_rows))
test_ds = Dataset.from_dict(expand_absa(test_rows))

In [8]:
from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding
)

In [9]:
model_name = "bert-base-multilingual-cased"
# model_name = "asafaya/bert-base-arabic"
# model_name = "aubmindlab/bert-base-arabertv2"

In [10]:
tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/625 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/49.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/996k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

In [11]:
def tokenize_fn(batch):
    return tokenizer(
        batch["category"],
        batch["text"],
        truncation=True,
        padding=False,
        max_length=256,
    )

In [12]:
train_ds = train_ds.map(tokenize_fn, batched=True)
dev_ds   = dev_ds.map(tokenize_fn, batched=True)
test_ds  = test_ds.map(tokenize_fn, batched=True)

Map:   0%|          | 0/7635 [00:00<?, ? examples/s]

Map:   0%|          | 0/1122 [00:00<?, ? examples/s]

Map:   0%|          | 0/2158 [00:00<?, ? examples/s]

In [13]:
model = AutoModelForSequenceClassification.from_pretrained(
    model_name,
    num_labels=len(LABELS),
    id2label=id2label,
    label2id=label2id,
)

model.safetensors:   0%|          | 0.00/714M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-multilingual-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [14]:
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [15]:
from sklearn.metrics import f1_score, accuracy_score, classification_report
def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    acc = accuracy_score(labels, preds)
    macro = f1_score(labels, preds, average="macro")
    return {"acc": acc, "macro_f1": macro}

In [16]:
out_dir = "/content/drive/MyDrive/FYP/absa/bert_mbert_conditioned_absa"

training_args = TrainingArguments(
    output_dir=out_dir,
    eval_strategy="epoch",
    save_strategy="epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=32,
    num_train_epochs=3,
    weight_decay=0.01,
    logging_steps=50,
    load_best_model_at_end=True,
    metric_for_best_model="macro_f1",
    greater_is_better=True,
    fp16=torch.cuda.is_available(),
    report_to="none",
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=dev_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [17]:
trainer.train()

Epoch,Training Loss,Validation Loss,Acc,Macro F1
1,0.537450,0.659647,0.793226,0.388487
2,0.481059,0.646085,0.805704,0.389514
3,0.395152,0.595327,0.811052,0.397782


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

TrainOutput(global_step=1434, training_loss=0.5030982896706383, metrics={'train_runtime': 111.7394, 'train_samples_per_second': 204.986, 'train_steps_per_second': 12.833, 'total_flos': 2374904100964824.0, 'train_loss': 0.5030982896706383, 'epoch': 3.0})

In [18]:
test_metrics = trainer.evaluate(eval_dataset=test_ds)

In [19]:
trainer.save_model(out_dir)
tokenizer.save_pretrained(out_dir)

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

('/content/drive/MyDrive/FYP/absa/bert_mbert_conditioned_absa/tokenizer_config.json',
 '/content/drive/MyDrive/FYP/absa/bert_mbert_conditioned_absa/tokenizer.json')

In [20]:
result = {
    "acc": test_metrics["eval_acc"],
    "macro_f1": test_metrics["eval_macro_f1"],
    "n": len(test_ds),
}

In [21]:
save_path = "/content/drive/MyDrive/FYP/absa/bert_results/mbert_conditioned_absa_results.json"
os.makedirs(os.path.dirname(save_path), exist_ok=True)
with open(save_path, "w", encoding="utf-8") as f:
    json.dump(result, f, ensure_ascii=False, indent=4)